<a href="https://colab.research.google.com/github/Devadeth-cmyk/vehicle-damage-insurance-ai/blob/development/damage_detection_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#Libraries

In [ ]:
!pip install ultralytics opencv-python-headless -q
import glob
import json
import os
from collections import Counter
import shutil
from tqdm import tqdm
from PIL import Image
import matplotlib.pyplot as plt
import random

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 8.4 MB/s eta 0:00:00


#Read Data

In [ ]:
dataset_path = "/content/drive/MyDrive/AI ML/Datasets/CarDD_COCO"

!find "{dataset_path}" -maxdepth 3 -type f | head -100  #Lists up to 100 files within the dataset directory

/content/drive/MyDrive/AI ML/Datasets/CarDD_COCO/annotations/instances_train2017.json
/content/drive/MyDrive/AI ML/Datasets/CarDD_COCO/annotations/instances_test2017.json
/content/drive/MyDrive/AI ML/Datasets/CarDD_COCO/annotations/instances_val2017.json
/content/drive/MyDrive/AI ML/Datasets/CarDD_COCO/annotations/image_info.xlsx
/content/drive/MyDrive/AI ML/Datasets/CarDD_COCO/test2017/000237.jpg
/content/drive/MyDrive/AI ML/Datasets/CarDD_COCO/test2017/000191.jpg
/content/drive/MyDrive/AI ML/Datasets/CarDD_COCO/test2017/000204.jpg
/content/drive/MyDrive/AI ML/Datasets/CarDD_COCO/test2017/000185.jpg
/content/drive/MyDrive/AI ML/Datasets/CarDD_COCO/test2017/000288.jpg
/content/drive/MyDrive/AI ML/Datasets/CarDD_COCO/test2017/000233.jpg
/content/drive/MyDrive/AI ML/Datasets/CarDD_COCO/test2017/000297.jpg
/content/drive/MyDrive/AI ML/Datasets/CarDD_COCO/test2017/000088.jpg
/content/drive/MyDrive/AI ML/Datasets/CarDD_COCO/test2017/000128.jpg
/content/drive/MyDrive/AI ML/Datasets/CarDD_COC

In [ ]:
# Used to finds all json files inside the dataset folder and prints its path
json_files = glob.glob(
    dataset_path + "/**/*.json",
    recursive=True
)

print("Found JSON files:", len(json_files))

for f in json_files:
    print(f)

Found JSON files: 3
/content/drive/MyDrive/AI ML/Datasets/CarDD_COCO/annotations/instances_train2017.json
/content/drive/MyDrive/AI ML/Datasets/CarDD_COCO/annotations/instances_test2017.json
/content/drive/MyDrive/AI ML/Datasets/CarDD_COCO/annotations/instances_val2017.json


In [ ]:
# Loads COCO annotation files and display basic informations like number of images, annotations and categories.
annotation_path = os.path.join(
    dataset_path,
    "annotations",
    "instances_train2017.json"
)

with open(annotation_path, "r") as f:
    coco_data = json.load(f)

print("Number of images:", len(coco_data["images"]))
print("Number of annotations:", len(coco_data["annotations"]))
print("Number of categories:", len(coco_data["categories"]))

print("\nCARDD Classes:")
for category in coco_data["categories"]:
    print(
        f'ID: {category["id"]} -> {category["name"]}'
    )

Number of images: 2816
Number of annotations: 6211
Number of categories: 6

CARDD Classes:
ID: 1 -> dent
ID: 2 -> scratch
ID: 3 -> crack
ID: 4 -> glass shatter
ID: 5 -> lamp broken
ID: 6 -> tire flat


In [ ]:
# It checks the total number of files and number of files in each train, val and split dataset
for split in ["train2017", "val2017", "test2017"]:
    split_path = os.path.join(dataset_path, split)

    files = os.listdir(split_path)

    print(f"\n{split}:")
    print("Number of files:", len(files)) # Used to display the first 5 files of each dataset
    print("First 5 files:")

    for file in files[:5]:
        print(" ", file)


train2017:
Number of files: 2816
First 5 files:
  002650.jpg
  002585.jpg
  002623.jpg
  002616.jpg
  002608.jpg

val2017:
Number of files: 810
First 5 files:
  000017.jpg
  000016.jpg
  000024.jpg
  000013.jpg
  000086.jpg

test2017:
Number of files: 374
First 5 files:
  000237.jpg
  000191.jpg
  000204.jpg
  000185.jpg
  000288.jpg


In [ ]:
# Checks for any images without annotations
image_ids = {img["id"] for img in coco_data["images"]}

annotation_image_ids = {
    ann["image_id"]
    for ann in coco_data["annotations"]
}

print("Images in JSON:", len(image_ids))
print("Images with annotations:", len(annotation_image_ids))

missing_annotations = image_ids - annotation_image_ids

print("Images without annotations:", len(missing_annotations))

Images in JSON: 2816
Images with annotations: 2816
Images without annotations: 0


#Preprocessing

In [ ]:
# Create folders for storing the converted data, which is used for training, validation and testing
yolo_root = "/content/drive/MyDrive/AI ML/Projects/CARDD_YOLO"

for split in ["train", "val", "test"]:
    os.makedirs(f"{yolo_root}/images/{split}", exist_ok=True)
    os.makedirs(f"{yolo_root}/labels/{split}", exist_ok=True)

print(yolo_root)

/content/drive/MyDrive/AI ML/Projects/CARDD_YOLO


In [ ]:
# Converting COCO annotations to YOLO Format
dataset_path = "/content/drive/MyDrive/AI ML/Datasets/CarDD_COCO"

splits = {
    "train": "train2017",
    "val": "val2017",
    "test": "test2017"
}

# Get class information
with open(f"{dataset_path}/annotations/instances_train2017.json", "r") as f:
    coco_data = json.load(f)

categories = sorted(coco_data["categories"], key=lambda x: x["id"])

# COCO category ID to YOLO class ID
category_to_yolo = {
    cat["id"]: i for i, cat in enumerate(categories)
}

class_names = [cat["name"] for cat in categories]

print("Classes:")
for i, name in enumerate(class_names):
    print(i, name)

print("Number of classes:", len(class_names))

Classes:
0 dent
1 scratch
2 crack
3 glass shatter
4 lamp broken
5 tire flat
Number of classes: 6


In [ ]:
# COCO to YOLO preprocessing
def convert_coco_to_yolo(json_path, image_dir, output_image_dir, output_label_dir):

    with open(json_path, "r") as f:
        coco = json.load(f)

    # Map image ID to image information
    images = {
        img["id"]: img
        for img in coco["images"]
    }

    # Group annotations by image
    annotations_by_image = {}

    for ann in coco["annotations"]:
        image_id = ann["image_id"]
        annotations_by_image.setdefault(image_id, []).append(ann)

    converted = 0

    for image_id, img_info in tqdm(images.items()):

        filename = img_info["file_name"]
        width = img_info["width"]
        height = img_info["height"]

        # Source image
        src_image = os.path.join(image_dir, filename)

        # Destination image
        dst_image = os.path.join(output_image_dir, filename)

        # Copy image
        if os.path.exists(src_image):
            shutil.copy2(src_image, dst_image)

        # Label filename
        label_name = os.path.splitext(filename)[0] + ".txt"
        label_path = os.path.join(output_label_dir, label_name)

        yolo_lines = []

        for ann in annotations_by_image.get(image_id, []):

            # Skip invalid annotations
            if ann.get("iscrowd", 0) == 1:
                continue

            category_id = ann["category_id"]

            if category_id not in category_to_yolo:
                continue

            x, y, w, h = ann["bbox"]

            # Convert COCO's bounding box to YOLO's bounding box

            x_center = x + w / 2
            y_center = y + h / 2

            # Normalize
            x_center /= width
            y_center /= height
            w /= width
            h /= height

            class_id = category_to_yolo[category_id]

            yolo_lines.append(
                f"{class_id} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}"
            )

        # Write label file
        with open(label_path, "w") as f:
            f.write("\n".join(yolo_lines))

        converted += 1

    print(f"Converted {converted} images")

In [ ]:
# Converts the COCO to YOLO for all datasets including train, val, test
for split, coco_split in splits.items():

    json_path = f"{dataset_path}/annotations/instances_{coco_split}.json"

    image_dir = f"{dataset_path}/{coco_split}"

    output_image_dir = f"{yolo_root}/images/{split}"
    output_label_dir = f"{yolo_root}/labels/{split}"

    convert_coco_to_yolo(
        json_path,
        image_dir,
        output_image_dir,
        output_label_dir
    )

 78%|███████▊  | 2202/2816 [39:23<09:23,  1.09it/s]

In [ ]:
# Checking the dataset after conversion
yolo_root = "/content/drive/MyDrive/AI ML/Projects/CARDD_YOLO"

for split in ["train", "val", "test"]:
    images = os.listdir(f"{yolo_root}/images/{split}")
    labels = os.listdir(f"{yolo_root}/labels/{split}")

    print(f"{split}: images={len(images)}, labels={len(labels)}")

In [ ]:
PROJECT_ROOT = "/content/drive/MyDrive/AI ML/Projects/CARDD_YOLO"

TRAIN_IMAGES = f"{PROJECT_ROOT}/images/train"
VAL_IMAGES   = f"{PROJECT_ROOT}/images/val"
TEST_IMAGES  = f"{PROJECT_ROOT}/images/test"

TRAIN_LABELS = f"{PROJECT_ROOT}/labels/train"
VAL_LABELS   = f"{PROJECT_ROOT}/labels/val"
TEST_LABELS  = f"{PROJECT_ROOT}/labels/test"

print("Project:", PROJECT_ROOT)
print("Train images:", len(os.listdir(TRAIN_IMAGES)))
print("Val images:", len(os.listdir(VAL_IMAGES)))
print("Test images:", len(os.listdir(TEST_IMAGES)))

In [ ]:
json_path = "/content/drive/MyDrive/AI ML/Datasets/CarDD_COCO/annotations/instances_train2017.json"

with open(json_path, "r") as f:
    coco = json.load(f)
print("CarDD categories:")

for category in coco["categories"]:
    print(category["id"], "->", category["name"])

In [ ]:
# Current YOLO dataset
yolo_root = "/content/drive/MyDrive/AI ML/Projects/CARDD_YOLO"

# Get class names from the COCO annotations
ann_dir = "/content/drive/MyDrive/AI ML/Datasets/CarDD_COCO/annotations"

train_json = [
    f for f in os.listdir(ann_dir)
    if f.startswith("instances_train") and f.endswith(".json")
][0]

with open(os.path.join(ann_dir, train_json), "r") as f:
    coco = json.load(f)

# Sort categories by the YOLO class ID
names = [
    category["name"]
    for category in sorted(
        coco["categories"],
        key=lambda x: category_to_yolo[x["id"]]
    )
]

data = {
    "path": yolo_root,
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": names
}

yaml_path = os.path.join(yolo_root, "data.yaml")

with open(yaml_path, "w") as f:
    yaml.dump(data, f, sort_keys=False)

print("Created:", yaml_path)
print("\nContents:")
with open(yaml_path, "r") as f:
    print(f.read())

In [ ]:
print("data.yaml exists:", os.path.exists(yaml_path))

In [ ]:
label_files = glob.glob(
    "/content/drive/MyDrive/AI ML/Projects/CARDD_YOLO/labels/train/*.txt"
)

print("Training label files:", len(label_files))

for file in label_files[:5]:
    print("\nFile:", os.path.basename(file))
    with open(file, "r") as f:
        print(f.read()[:1000])

In [ ]:
image_files = glob.glob(
    "/content/drive/MyDrive/AI ML/Projects/CARDD_YOLO/images/train/*"
)

img_path = random.choice(image_files)
label_path = os.path.join(
    "/content/drive/MyDrive/AI ML/Projects/CARDD_YOLO/labels/train",
    os.path.splitext(os.path.basename(img_path))[0] + ".txt"
)

img = Image.open(img_path)
print("Image:", img_path)
print("Size:", img.size)

plt.figure(figsize=(10, 8))
plt.imshow(img)
plt.axis("off")
plt.show()

print("\nLabel:")
with open(label_path) as f:
    print(f.read())

#Inference
The CarDD dataset preprocessing was completed. The original COCO-format annotations were converted into YOLO-compatible label files, the dataset was organized into training, validation, and testing splits, and the required data.yaml configuration was created. The resulting dataset is ready for YOLO model training.